# Phase 2: Model Training and Comparison

This notebook trains and compares machine learning regression models for Phase 2 of the Halophyte Grass Dictionary final year project.

The dataset contains 30 grasses from the cleaned Phase 1 dataset. The prediction target is numeric salt tolerance and ion concentration factors, including GR50 average and shoot/root ion values.

Because this is a small biological dataset, the model is used for estimated prediction only. It should support academic exploration and application prototyping, but it should not be presented as perfect scientific certainty.

## 1. Notebook Setup

This section imports the libraries used for data loading, cleaning, visualization, model training, evaluation, prediction, and model export.

In [ ]:
from pathlib import Path
import json
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
warnings.filterwarnings('default')

## 2. Load Dataset

The notebook prefers a CSV dataset if one is available. It checks these CSV locations first:

- `ml/data/halophyte_grass_library.csv`
- `data/halophyte_grass_library.csv`
- `halophyte_grass_library.csv`

If a CSV is not available, it attempts to read `src/data/grassLibraryData.ts` or `grassLibraryData.ts` as a fallback.

If loading fails, place the cleaned CSV at one of the CSV paths above and re-run the notebook.

In [ ]:
def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'package.json').exists() and (candidate / 'ml').exists():
            return candidate
    if current.name.lower() == 'notebooks' and current.parent.name.lower() == 'ml':
        return current.parent.parent
    if current.name.lower() == 'notebooks':
        return current.parent
    return current

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')

CSV_CANDIDATES = [
    PROJECT_ROOT / 'ml' / 'data' / 'halophyte_grass_library.csv',
    PROJECT_ROOT / 'data' / 'halophyte_grass_library.csv',
    PROJECT_ROOT / 'halophyte_grass_library.csv',
]

TS_CANDIDATES = [
    PROJECT_ROOT / 'src' / 'data' / 'grassLibraryData.ts',
    PROJECT_ROOT / 'grassLibraryData.ts',
]


def load_from_typescript(ts_path):
    text = ts_path.read_text(encoding='utf-8')
    match = re.search(r'grassLibraryData[^=]*=\s*(\[.*?\n\]);', text, flags=re.DOTALL)
    if not match:
        raise ValueError(f'Could not locate grassLibraryData array in {ts_path}')
    json_text = match.group(1).strip()
    if json_text.endswith(';'):
        json_text = json_text[:-1]
    records = json.loads(json_text)
    return pd.DataFrame(records)


def load_dataset():
    for path in CSV_CANDIDATES:
        if path.exists():
            print(f'Loaded CSV dataset from: {path}')
            return pd.read_csv(path), path

    for path in TS_CANDIDATES:
        if path.exists():
            print(f'CSV not found. Loaded TypeScript fallback from: {path}')
            return load_from_typescript(path), path

    searched = [str(path) for path in CSV_CANDIDATES + TS_CANDIDATES]
    raise FileNotFoundError(
        'Cleaned dataset was not found. Place halophyte_grass_library.csv in ml/data/, data/, or the project root. '
        f'Searched: {searched}'
    )

raw_df, dataset_path = load_dataset()
raw_df.head()

## 3. Clean and Prepare Data

This section standardizes column names, converts numeric values safely, calculates GR50 min/max/average when necessary, and encodes the mechanism field:

- Salt-Secreting = 1
- Non-Secreting = 0

In [ ]:
def snake_case_column(name):
    name = str(name).strip().lower()
    replacements = {
        '+': '',
        '-1': '',
        'kg-1': 'kg',
        'mmol kg': 'mmol_kg',
        'cl?': 'cl',
        'cl-': 'cl',
        'na+': 'na',
        'k+': 'k',
        'gr50': 'gr50',
        '/': '_',
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = re.sub(r'[^a-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name


def clean_numeric(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).replace(',', '').replace('*', '').strip()
    numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
    if not numbers:
        return np.nan
    return float(numbers[0])


def parse_numeric_range(value):
    if pd.isna(value):
        return np.nan, np.nan, np.nan
    if isinstance(value, (int, float, np.number)):
        number = float(value)
        return number, number, number
    text = str(value).replace(',', '').replace('*', '').strip()
    numbers = [float(item) for item in re.findall(r'-?\d+(?:\.\d+)?', text)]
    if not numbers:
        return np.nan, np.nan, np.nan
    if len(numbers) == 1:
        return numbers[0], numbers[0], numbers[0]
    minimum = min(numbers[0], numbers[1])
    maximum = max(numbers[0], numbers[1])
    average = (minimum + maximum) / 2
    return minimum, maximum, average


def first_existing_column(df, aliases):
    for alias in aliases:
        clean_alias = snake_case_column(alias)
        if clean_alias in df.columns:
            return clean_alias
    return None


def normalize_mechanism(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace('_', '-').replace(' ', '-')
    if text.startswith('non'):
        return 'Non-Secreting'
    if 'salt' in text and 'secreting' in text:
        return 'Salt-Secreting'
    if text in {'1', 'true', 'yes'}:
        return 'Salt-Secreting'
    if text in {'0', 'false', 'no'}:
        return 'Non-Secreting'
    return str(value).strip()


def require_column(df, canonical_name, aliases):
    column = first_existing_column(df, aliases)
    if column is None:
        raise KeyError(f'Missing required column for {canonical_name}. Tried aliases: {aliases}')
    return column

prepared_raw_df = raw_df.copy()
prepared_raw_df.columns = [snake_case_column(column) for column in prepared_raw_df.columns]
print('Cleaned source columns:')
print(prepared_raw_df.columns.tolist())

COLUMN_ALIASES = {
    'species': ['species', 'grass_species', 'scientific_name', 'species_full_name'],
    'common_name': ['common_name', 'common'],
    'mechanism': ['mechanism', 'salt_mechanism'],
    'gr50_min': ['gr50_min', 'gr50_min_ds_m', 'gr50_minimum'],
    'gr50_max': ['gr50_max', 'gr50_max_ds_m', 'gr50_maximum'],
    'gr50_avg': ['gr50_avg', 'gr50_average', 'gr50_avg_ds_m'],
    'gr50_range': ['gr50', 'gr50_display', 'gr50_range'],
    'na_shoot': ['na_shoot', 'na_shoot_mmol_kg_dw', 'na_shoot_content'],
    'na_root': ['na_root', 'na_root_mmol_kg_dw', 'na_root_content'],
    'cl_shoot': ['cl_shoot', 'cl_shoot_mmol_kg_dw', 'cl_shoot_content'],
    'cl_root': ['cl_root', 'cl_root_mmol_kg_dw', 'cl_root_content'],
    'k_shoot': ['k_shoot', 'k_shoot_mmol_kg_dw', 'k_shoot_content'],
    'k_root': ['k_root', 'k_root_mmol_kg_dw', 'k_root_content'],
}

species_col = require_column(prepared_raw_df, 'species', COLUMN_ALIASES['species'])
common_name_col = require_column(prepared_raw_df, 'common_name', COLUMN_ALIASES['common_name'])
mechanism_col = require_column(prepared_raw_df, 'mechanism', COLUMN_ALIASES['mechanism'])

clean_df = pd.DataFrame({
    'species': prepared_raw_df[species_col].astype(str).str.strip(),
    'common_name': prepared_raw_df[common_name_col].astype(str).str.strip(),
    'mechanism': prepared_raw_df[mechanism_col].apply(normalize_mechanism),
})

for canonical in ['na_shoot', 'na_root', 'cl_shoot', 'cl_root', 'k_shoot', 'k_root']:
    source_col = require_column(prepared_raw_df, canonical, COLUMN_ALIASES[canonical])
    clean_df[canonical] = prepared_raw_df[source_col].apply(clean_numeric)

# Prefer explicit GR50 columns. If they are missing, calculate min, max, and average from a range column.
gr50_min_col = first_existing_column(prepared_raw_df, COLUMN_ALIASES['gr50_min'])
gr50_max_col = first_existing_column(prepared_raw_df, COLUMN_ALIASES['gr50_max'])
gr50_avg_col = first_existing_column(prepared_raw_df, COLUMN_ALIASES['gr50_avg'])
gr50_range_col = first_existing_column(prepared_raw_df, COLUMN_ALIASES['gr50_range'])

if gr50_min_col and gr50_max_col:
    clean_df['gr50_min'] = prepared_raw_df[gr50_min_col].apply(clean_numeric)
    clean_df['gr50_max'] = prepared_raw_df[gr50_max_col].apply(clean_numeric)
else:
    if gr50_range_col is None:
        raise KeyError('GR50 columns are missing. Provide gr50_min/gr50_max/gr50_avg or a GR50 range/display column.')
    parsed_ranges = prepared_raw_df[gr50_range_col].apply(parse_numeric_range)
    clean_df['gr50_min'] = parsed_ranges.apply(lambda item: item[0])
    clean_df['gr50_max'] = parsed_ranges.apply(lambda item: item[1])

if gr50_avg_col:
    clean_df['gr50_avg'] = prepared_raw_df[gr50_avg_col].apply(clean_numeric)
else:
    clean_df['gr50_avg'] = clean_df[['gr50_min', 'gr50_max']].mean(axis=1)

MECHANISM_ENCODER = {'Non-Secreting': 0, 'Salt-Secreting': 1}
clean_df['mechanism_encoded'] = clean_df['mechanism'].map(MECHANISM_ENCODER)

NUMERIC_FIELDS = [
    'gr50_avg',
    'na_shoot',
    'na_root',
    'cl_shoot',
    'cl_root',
    'k_shoot',
    'k_root',
]

CONTEXT_COLUMNS = ['species', 'common_name', 'mechanism']
MODEL_COLUMNS = CONTEXT_COLUMNS + ['mechanism_encoded', 'gr50_min', 'gr50_max'] + NUMERIC_FIELDS
clean_df = clean_df[MODEL_COLUMNS]

if clean_df['mechanism_encoded'].isna().any():
    raise ValueError('Some mechanism values could not be encoded. Check the mechanism column values.')

clean_df.head()

## 4. Exploratory Data Analysis

The charts are intentionally simple and academic. They help inspect the dataset before training models.

In [ ]:
print(f'Dataset shape: {clean_df.shape}')
clean_df.head()

In [ ]:
print('Mechanism counts:')
clean_df['mechanism'].value_counts()

In [ ]:
print('Missing values by column:')
clean_df.isna().sum()

In [ ]:
print('Summary statistics for numeric fields:')
clean_df[NUMERIC_FIELDS].describe().T

In [ ]:
correlation = clean_df[NUMERIC_FIELDS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC_FIELDS)))
ax.set_yticks(range(len(NUMERIC_FIELDS)))
ax.set_xticklabels(NUMERIC_FIELDS, rotation=45, ha='right')
ax.set_yticklabels(NUMERIC_FIELDS)
ax.set_title('Correlation Matrix for Numeric Salt Tolerance and Ion Fields')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
gr50_by_mechanism = clean_df.groupby('mechanism')['gr50_avg'].mean().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
gr50_by_mechanism.plot(kind='bar', ax=ax, color=['#4c78a8', '#f58518'])
ax.set_title('Average GR50 by Mechanism')
ax.set_xlabel('Mechanism')
ax.set_ylabel('Average GR50 (dS/m)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
ion_summary_df = clean_df.copy()
ion_summary_df['na_average'] = ion_summary_df[['na_shoot', 'na_root']].mean(axis=1)
ion_summary_df['cl_average'] = ion_summary_df[['cl_shoot', 'cl_root']].mean(axis=1)
ion_summary_df['k_average'] = ion_summary_df[['k_shoot', 'k_root']].mean(axis=1)

ion_by_mechanism = ion_summary_df.groupby('mechanism')[['na_average', 'cl_average', 'k_average']].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ion_by_mechanism.plot(kind='bar', ax=ax)
ax.set_title('Average Na, Cl, and K Values by Mechanism')
ax.set_xlabel('Mechanism')
ax.set_ylabel('Average ion value')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Ion group')
plt.tight_layout()
plt.show()

## 5. Define the Machine Learning Problem

We are treating this as a regression problem, not a classification problem.

The Phase 2 app use case is: a student may enter one known numerical factor and expect estimated values for the remaining fields. Therefore, this notebook tests models that predict multiple output values.

The seven main numeric fields are:

- `gr50_avg`
- `na_shoot`
- `na_root`
- `cl_shoot`
- `cl_root`
- `k_shoot`
- `k_root`

For evaluation, each numeric field is treated as the known input field once. The model receives `mechanism_encoded` plus that known field, then predicts the other six fields.

Example: if known input = `na_shoot`, the input features are `mechanism_encoded` and `na_shoot`, while the targets are `gr50_avg`, `na_root`, `cl_shoot`, `cl_root`, `k_shoot`, and `k_root`.

## 6. Train and Compare Regression Models

The models compared here are intentionally simple and supervisor-friendly:

- KNeighborsRegressor with 3 neighbors
- KNeighborsRegressor with 5 neighbors
- RandomForestRegressor
- Ridge Regression
- DecisionTreeRegressor

KNN and Ridge use feature scaling through a pipeline. Tree-based models do not require scaling.

In [ ]:
def make_model(model_name):
    if model_name == 'knn_3':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', KNeighborsRegressor(n_neighbors=3)),
        ])
    if model_name == 'knn_5':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', KNeighborsRegressor(n_neighbors=5)),
        ])
    if model_name == 'random_forest':
        return RandomForestRegressor(n_estimators=100, random_state=42)
    if model_name == 'ridge_regression':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', Ridge()),
        ])
    if model_name == 'decision_tree':
        return DecisionTreeRegressor(random_state=42)
    raise ValueError(f'Unknown model_name: {model_name}')

MODEL_NAMES = ['knn_3', 'knn_5', 'random_forest', 'ridge_regression', 'decision_tree']

def target_fields_for(known_field):
    if known_field not in NUMERIC_FIELDS:
        raise ValueError(f'known_field must be one of {NUMERIC_FIELDS}')
    return [field for field in NUMERIC_FIELDS if field != known_field]

## 7. Evaluation Method: Leave-One-Out Cross Validation

Because the dataset has only 30 records, this notebook uses Leave-One-Out Cross Validation instead of relying on a normal train/test split.

For each known input field and each model, the notebook calculates:

- MAE: Mean Absolute Error
- RMSE: Root Mean Squared Error
- R2 score: regression fit score

This is regression, so we do not call the result "accuracy". The closest equivalent is R2 score, while MAE and RMSE show prediction error in the units of the target variables.

In [ ]:
def evaluate_model_for_known_field(model_name, known_field):
    target_fields = target_fields_for(known_field)
    required_columns = ['mechanism_encoded', known_field] + target_fields
    evaluation_df = clean_df.dropna(subset=required_columns).copy()

    if len(evaluation_df) < 6:
        raise ValueError(f'Not enough complete rows to evaluate {model_name} for known field {known_field}.')

    X = evaluation_df[['mechanism_encoded', known_field]]
    y = evaluation_df[target_fields]

    loo = LeaveOneOut()
    predictions = []
    actuals = []

    for train_index, test_index in loo.split(X):
        model = make_model(model_name)
        model.fit(X.iloc[train_index], y.iloc[train_index])
        prediction = model.predict(X.iloc[test_index])
        predictions.append(prediction[0])
        actuals.append(y.iloc[test_index].to_numpy()[0])

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    return {
        'known_input_field': known_field,
        'model_name': model_name,
        'target_fields': ', '.join(target_fields),
        'average_mae': mean_absolute_error(actuals, predictions),
        'average_rmse': float(np.sqrt(mean_squared_error(actuals, predictions))),
        'average_r2': r2_score(actuals, predictions, multioutput='uniform_average'),
        'records_used': len(evaluation_df),
    }

comparison_rows = []
for known_field in NUMERIC_FIELDS:
    for model_name in MODEL_NAMES:
        comparison_rows.append(evaluate_model_for_known_field(model_name, known_field))

comparison_results = pd.DataFrame(comparison_rows)
comparison_results = comparison_results.sort_values(['known_input_field', 'average_rmse']).reset_index(drop=True)
comparison_results

In [ ]:
best_by_known_input = (
    comparison_results
    .sort_values(['known_input_field', 'average_rmse'])
    .groupby('known_input_field', as_index=False)
    .first()
)

ranked_best_by_known_input = best_by_known_input[[
    'known_input_field',
    'model_name',
    'target_fields',
    'average_mae',
    'average_rmse',
    'average_r2',
]]

ranked_best_by_known_input

In [ ]:
overall_model_ranking = (
    comparison_results
    .groupby('model_name', as_index=False)[['average_mae', 'average_rmse', 'average_r2']]
    .mean()
    .sort_values('average_rmse')
    .reset_index(drop=True)
)

overall_model_ranking

## 8. Select Models for Prediction

The table above shows the metric-best model per known input field based on lowest RMSE.

For app integration, KNN is especially useful because it can also show similar grasses used for prediction. Therefore, this notebook prefers `knn_3` when its RMSE is within 10 percent of the metric-best model for a known field. If it is not close, the metric-best model is selected.

In [ ]:
selection_rows = []
selected_model_per_known_field = {}
metric_best_model_per_known_field = {}

for known_field in NUMERIC_FIELDS:
    field_results = comparison_results[comparison_results['known_input_field'] == known_field].copy()
    metric_best = field_results.sort_values('average_rmse').iloc[0]
    knn3_row = field_results[field_results['model_name'] == 'knn_3'].iloc[0]

    metric_best_model_per_known_field[known_field] = metric_best['model_name']

    if knn3_row['average_rmse'] <= metric_best['average_rmse'] * 1.10:
        selected_model = 'knn_3'
        reason = 'knn_3 is within 10 percent of the best RMSE and supports similar grass lookup.'
    else:
        selected_model = metric_best['model_name']
        reason = 'Metric-best model selected because knn_3 was not close enough by RMSE.'

    selected_model_per_known_field[known_field] = selected_model
    selected_row = field_results[field_results['model_name'] == selected_model].iloc[0]
    selection_rows.append({
        'known_input_field': known_field,
        'metric_best_model': metric_best['model_name'],
        'metric_best_rmse': metric_best['average_rmse'],
        'selected_model_for_app': selected_model,
        'selected_model_rmse': selected_row['average_rmse'],
        'selection_reason': reason,
    })

selected_models_summary = pd.DataFrame(selection_rows)
selected_models_summary

## 9. Train Final Models on Complete Available Data

After evaluation, final models are trained on all complete rows available for each known input field. These trained models are used by the prediction function and saved as artifacts for later backend integration.

In [ ]:
trained_models = {}
training_data_by_known_field = {}
target_fields_by_known_input = {}

for known_field in NUMERIC_FIELDS:
    target_fields = target_fields_for(known_field)
    required_columns = ['mechanism_encoded', known_field] + target_fields
    train_df = clean_df.dropna(subset=required_columns).copy().reset_index(drop=True)

    if len(train_df) < 6:
        raise ValueError(f'Not enough complete rows to train final models for known field {known_field}.')

    X_train = train_df[['mechanism_encoded', known_field]]
    y_train = train_df[target_fields]

    trained_models[known_field] = {}
    target_fields_by_known_input[known_field] = target_fields
    training_data_by_known_field[known_field] = {
        'rows': train_df,
        'X': X_train,
        'y': y_train,
    }

    for model_name in MODEL_NAMES:
        model = make_model(model_name)
        model.fit(X_train, y_train)
        trained_models[known_field][model_name] = model

print('Final models trained for known input fields:')
print(list(trained_models.keys()))

## 10. Prediction Function

The reusable function below supports two modes:

1. `grass_based`: user provides `species`, `known_field`, and `known_value`. The function uses the species to infer mechanism context.
2. `mechanism_based`: user provides `mechanism`, `known_field`, and `known_value`. The function uses the selected mechanism group context.

For KNN predictions, the function also returns up to 3 similar grasses used by the neighbor model.

In [ ]:
def normalize_known_field(field_name):
    field_name = snake_case_column(field_name)
    aliases = {
        'gr50_average': 'gr50_avg',
        'gr50_avg_ds_m': 'gr50_avg',
        'na_shoot_mmol_kg_dw': 'na_shoot',
        'na_root_mmol_kg_dw': 'na_root',
        'cl_shoot_mmol_kg_dw': 'cl_shoot',
        'cl_root_mmol_kg_dw': 'cl_root',
        'k_shoot_mmol_kg_dw': 'k_shoot',
        'k_root_mmol_kg_dw': 'k_root',
    }
    return aliases.get(field_name, field_name)


def find_species_row(species):
    if species is None:
        raise ValueError('species is required when mode="grass_based".')
    query = str(species).strip().lower()
    exact_match = clean_df[clean_df['species'].str.lower() == query]
    if not exact_match.empty:
        return exact_match.iloc[0]

    contains_match = clean_df[
        clean_df['species'].str.lower().str.contains(re.escape(query), na=False)
        | clean_df['common_name'].str.lower().str.contains(re.escape(query), na=False)
    ]
    if not contains_match.empty:
        return contains_match.iloc[0]

    raise ValueError(f'Species not found: {species}')


def mechanism_to_encoded(mechanism):
    normalized = normalize_mechanism(mechanism)
    if normalized not in MECHANISM_ENCODER:
        raise ValueError(f'Unknown mechanism: {mechanism}. Expected Salt-Secreting or Non-Secreting.')
    return normalized, MECHANISM_ENCODER[normalized]


def get_similar_grasses_for_knn(known_field, known_value, mechanism_encoded, model_name):
    if not model_name.startswith('knn'):
        return []

    model = trained_models[known_field][model_name]
    train_info = training_data_by_known_field[known_field]
    train_rows = train_info['rows']

    query_df = pd.DataFrame([[mechanism_encoded, float(known_value)]], columns=['mechanism_encoded', known_field])

    if isinstance(model, Pipeline):
        scaler = model.named_steps['scaler']
        knn_model = model.named_steps['model']
        query_values = scaler.transform(query_df)
    else:
        knn_model = model
        query_values = query_df

    neighbor_count = min(3, len(train_rows))
    distances, indices = knn_model.kneighbors(query_values, n_neighbors=neighbor_count)

    similar_grasses = []
    for distance, row_position in zip(distances[0], indices[0]):
        row = train_rows.iloc[int(row_position)]
        similar_grasses.append({
            'species': row['species'],
            'common_name': row['common_name'],
            'mechanism': row['mechanism'],
            'distance_or_similarity_note': f'scaled feature distance: {float(distance):.3f}',
        })
    return similar_grasses


def predict_remaining_values(
    mode,
    known_field,
    known_value,
    species=None,
    mechanism=None,
    model_name='knn_3',
):
    known_field = normalize_known_field(known_field)
    if known_field not in NUMERIC_FIELDS:
        raise ValueError(f'known_field must be one of {NUMERIC_FIELDS}')

    known_value = clean_numeric(known_value)
    if pd.isna(known_value):
        raise ValueError('known_value must contain a numeric value.')

    mode = str(mode).strip().lower()

    if mode == 'grass_based':
        species_row = find_species_row(species)
        selected_species = species_row['species']
        selected_common_name = species_row['common_name']
        selected_mechanism = species_row['mechanism']
        selected_mechanism, mechanism_encoded = mechanism_to_encoded(selected_mechanism)
        selected_context = selected_species
    elif mode == 'mechanism_based':
        selected_species = None
        selected_common_name = None
        selected_mechanism, mechanism_encoded = mechanism_to_encoded(mechanism)
        selected_context = selected_mechanism
    else:
        raise ValueError('mode must be either "grass_based" or "mechanism_based".')

    if model_name in {'best', 'selected'}:
        resolved_model_name = selected_model_per_known_field[known_field]
    else:
        resolved_model_name = model_name

    if resolved_model_name not in trained_models[known_field]:
        available = list(trained_models[known_field].keys()) + ['best', 'selected']
        raise ValueError(f'Unknown model_name for {known_field}: {model_name}. Available: {available}')

    target_fields = target_fields_by_known_input[known_field]
    model = trained_models[known_field][resolved_model_name]
    prediction_input = pd.DataFrame([[mechanism_encoded, known_value]], columns=['mechanism_encoded', known_field])
    predicted_values = model.predict(prediction_input)[0]

    predicted_remaining_values = {
        target: round(float(value), 3)
        for target, value in zip(target_fields, predicted_values)
    }

    similar_grasses = get_similar_grasses_for_knn(
        known_field=known_field,
        known_value=known_value,
        mechanism_encoded=mechanism_encoded,
        model_name=resolved_model_name,
    )

    return {
        'input_mode': mode,
        'selected_species': selected_species,
        'selected_common_name': selected_common_name,
        'selected_mechanism': selected_mechanism,
        'selected_context': selected_context,
        'known_field': known_field,
        'known_value': float(known_value),
        'predicted_remaining_values': predicted_remaining_values,
        'similar_grasses_used': similar_grasses,
        'model_name': resolved_model_name,
        'note': 'Prediction is an estimate based on a limited 30-grass dataset and should not be treated as perfect scientific certainty.',
    }

## 11. Save Best Model Artifacts

The artifact folder is `ml/models/`. The saved bundle contains the trained models, selected model per known field, scaler information through model pipelines, numeric field list, mechanism mapping, cleaned dataframe, and lookup data needed for similar grasses.

In [ ]:
ARTIFACT_DIR = PROJECT_ROOT / 'ml' / 'models'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

best_model_bundle = {
    'trained_models': trained_models,
    'selected_best_model_per_known_field': selected_model_per_known_field,
    'metric_best_model_per_known_field': metric_best_model_per_known_field,
    'target_fields_by_known_input': target_fields_by_known_input,
    'numeric_fields': NUMERIC_FIELDS,
    'context_columns': CONTEXT_COLUMNS,
    'mechanism_encoder_mapping': MECHANISM_ENCODER,
    'original_cleaned_dataframe': clean_df,
    'training_data_by_known_field': training_data_by_known_field,
    'model_names': MODEL_NAMES,
    'scaler_note': 'KNN and Ridge models use StandardScaler inside their sklearn Pipeline objects.',
    'dataset_path': str(dataset_path),
    'prediction_note': 'Predictions are estimates based on a limited 30-grass dataset.',
}

joblib.dump(best_model_bundle, ARTIFACT_DIR / 'best_model_bundle.joblib')
comparison_results.to_csv(ARTIFACT_DIR / 'model_comparison_results.csv', index=False)

metrics_summary = {
    'record_count': int(len(clean_df)),
    'numeric_fields': NUMERIC_FIELDS,
    'mechanism_encoder_mapping': MECHANISM_ENCODER,
    'overall_model_ranking': overall_model_ranking.to_dict(orient='records'),
    'best_model_per_known_input_by_lowest_rmse': ranked_best_by_known_input.to_dict(orient='records'),
    'selected_model_per_known_input_for_app': selected_models_summary.to_dict(orient='records'),
    'selection_policy': 'Prefer knn_3 when its RMSE is within 10 percent of the metric-best model; otherwise use the metric-best model.',
    'evaluation_method': 'Leave-One-Out Cross Validation',
    'metrics': ['MAE', 'RMSE', 'R2 score'],
}

with open(ARTIFACT_DIR / 'model_metrics_summary.json', 'w', encoding='utf-8') as file:
    json.dump(metrics_summary, file, indent=2)

with open(ARTIFACT_DIR / 'numeric_fields.json', 'w', encoding='utf-8') as file:
    json.dump({'numeric_fields': NUMERIC_FIELDS}, file, indent=2)

print(f'Saved model artifacts to: {ARTIFACT_DIR}')
print('Created files:')
for artifact_path in sorted(ARTIFACT_DIR.iterdir()):
    print(f'- {artifact_path.name}')

## 12. Quick Prediction Tests

These examples check that both prediction modes work from top to bottom.

In [ ]:
example_species = clean_df.iloc[0]['species']

example_1 = predict_remaining_values(
    mode='grass_based',
    species=example_species,
    known_field='gr50_avg',
    known_value=50,
    model_name='knn_3',
)

print('Example 1: grass_based prediction')
print(json.dumps(example_1, indent=2))

In [ ]:
example_2 = predict_remaining_values(
    mode='mechanism_based',
    mechanism='Salt-Secreting',
    known_field='na_shoot',
    known_value=1000,
    model_name='knn_3',
)

print('Example 2: mechanism_based prediction')
print(json.dumps(example_2, indent=2))

## 13. Final Conclusion

This notebook compared KNN, Random Forest, Ridge Regression, and Decision Tree regression models using Leave-One-Out Cross Validation.

The best model overall should be selected from `overall_model_ranking` using the lowest average RMSE. The metric-best model per known input field is shown in `ranked_best_by_known_input`.

For app integration, KNN is preferred when its metrics are close to the best model because KNN can also show similar grasses used for prediction. This is helpful for explaining predictions to students and supervisors.

Because the dataset contains only 30 grasses, predictions should always be displayed as estimates. The model can support missing-value estimation and educational exploration, but it should not be presented as a replacement for controlled biological experiments.

Next step after notebook validation: integrate `ml/models/best_model_bundle.joblib` into the backend/API. The FastAPI backend should be built only after this notebook and its saved artifacts are reviewed.